# J&J MedTech Sales Genie — Workshop

This notebook runs the full workshop end-to-end. Complete the three manual prerequisites below, fill in the widgets, then **Run All**.

---

## Prerequisites (manual — do these first)

### 1 — Note your SQL Warehouse ID

1. In the left sidebar, go to **SQL Warehouses**
2. Open any running warehouse (or start one)
3. Copy the **ID** from the URL or the warehouse detail page
   - It looks like: `6f538f22f07fe0d8`
4. Paste it into the **SQL Warehouse ID** widget below

### 2 — Create a Genie Space and note its ID

1. In the left sidebar, go to **Genie**
2. Click **New Genie Space** and give it any name (e.g. "MedTech Sales")
3. Save it — you don't need to configure it yet; the workshop will update it
4. Copy the **Space ID** from the URL: `.../genie/spaces/<space-id>`
5. Paste it into the **Genie Space ID** widget below

### 3 — Create a Databricks App (no deployment needed)

1. In the left sidebar, go to **Apps**
2. Click **Create App**
3. Give it a name using only lowercase letters, numbers, and hyphens (e.g. `my-first-genie-app`)
4. Click **Create** — do **not** deploy yet; the workshop will handle deployment
5. Enter that name in the **App Name** widget below

---

Everything else — catalog, schema, volume, tables, metric views, and app deployment — is handled automatically by the cells below.

In [ ]:
dbutils.widgets.text("catalog",        "medtech",            "Catalog")
dbutils.widgets.text("schema",         "sales",              "Schema")
dbutils.widgets.text("volume_name",    "raw_data",           "Volume Name")
dbutils.widgets.text("warehouse_id",   "",                   "SQL Warehouse ID")
dbutils.widgets.text("genie_space_id", "",                   "Genie Space ID")
dbutils.widgets.text("app_name",       "my-first-genie-app", "App Name (lowercase, hyphens only)")

catalog        = dbutils.widgets.get("catalog")
schema         = dbutils.widgets.get("schema")
volume_name    = dbutils.widgets.get("volume_name")
warehouse_id   = dbutils.widgets.get("warehouse_id")
genie_space_id = dbutils.widgets.get("genie_space_id")
app_name       = dbutils.widgets.get("app_name")

if not warehouse_id:
    raise ValueError("warehouse_id is required — see Prerequisites step 1")
if not genie_space_id:
    raise ValueError("genie_space_id is required — see Prerequisites step 2")

print(f"catalog:        {catalog}")
print(f"schema:         {schema}")
print(f"volume_name:    {volume_name}")
print(f"warehouse_id:   {warehouse_id}")
print(f"genie_space_id: {genie_space_id}")
print(f"app_name:       {app_name}")

## Step 1 — Setup and Load Raw Data

Creates catalog, schema, and volume; copies the CSV files into the volume; loads 3 Delta tables.

In [ ]:
dbutils.notebook.run("01_setup_and_load", timeout_seconds=600, arguments={
    "catalog":     catalog,
    "schema":      schema,
    "volume_name": volume_name,
})
print("Step 1 complete")

## Step 2 — Add Unity Catalog Metadata

Adds primary keys, table and column comments.

In [ ]:
dbutils.notebook.run("02_add_uc_metadata", timeout_seconds=300, arguments={
    "catalog": catalog,
    "schema":  schema,
})
print("Step 2 complete")

## Step 3 — Add Business Semantics

Creates 7 governed metric views.

In [ ]:
dbutils.notebook.run("03_add_business_semantics", timeout_seconds=300, arguments={
    "catalog": catalog,
    "schema":  schema,
})
print("Step 3 complete")

## Step 4 — Configure Genie Space

Updates the Genie Space with tables, metric views, and instructions.

In [ ]:
dbutils.notebook.run("04_create_genie_space", timeout_seconds=120, arguments={
    "catalog":        catalog,
    "schema":         schema,
    "warehouse_id":   warehouse_id,
    "genie_space_id": genie_space_id,
})
print("Step 4 complete")

## Step 5 — Deploy App

Writes `app.yaml`, deploys the Databricks App, grants permissions, and prints the app URL.

In [ ]:
dbutils.notebook.run("05_deploy_app", timeout_seconds=600, arguments={
    "app_name":       app_name,
    "warehouse_id":   warehouse_id,
    "genie_space_id": genie_space_id,
    "catalog":        catalog,
    "schema":         schema,
})
print("Step 5 complete")